# Random Projections and Johnson-Lindenstrauss Lemma

## Learning Objectives

1. **State** the Johnson-Lindenstrauss (JL) lemma
2. **Implement** Gaussian and sparse random projections
3. **Verify** distance preservation experimentally
4. **Compute** the minimum projection dimension for given $\epsilon, n$

## The Johnson-Lindenstrauss Lemma

**Theorem (JL, 1984):** For any set of $n$ points $X \subset \mathbb{R}^d$ and $\epsilon \in (0,1)$, there exists a map $f: \mathbb{R}^d \to \mathbb{R}^k$ such that for all $x, y \in X$:

$$(1-\epsilon)\|x-y\|^2 \leq \|f(x)-f(y)\|^2 \leq (1+\epsilon)\|x-y\|^2$$

with:
$$k \geq \frac{4 \ln n}{\epsilon^2/2 - \epsilon^3/3}$$

**Key insight:** $k$ depends only on $n$ (number of points) and $\epsilon$, **not on $d$**.

This means we can reduce from millions of dimensions to a few hundred with controlled distortion.

In [1]:
import math, random

def jl_dimension(n, epsilon):
    """Minimum projection dimension for JL lemma."""
    return math.ceil(4 * math.log(n) / (epsilon**2/2 - epsilon**3/3))

print("Required projection dimension k:")
print(f"{'n':>10} {'ε=0.1':>8} {'ε=0.2':>8} {'ε=0.3':>8}")
for n in [100, 1000, 10000, 100000]:
    row = [jl_dimension(n, eps) for eps in [0.1, 0.2, 0.3]]
    print(f"{n:>10} {row[0]:>8} {row[1]:>8} {row[2]:>8}")

print("""
Note: k is logarithmic in n, independent of d!""")

Required projection dimension k:
         n    ε=0.1    ε=0.2    ε=0.3
       100     3948     1063      512
      1000     5921     1595      768
     10000     7895     2126     1024
    100000     9869     2657     1280

Note: k is logarithmic in n, independent of d!


In [2]:
def norm(v): return math.sqrt(sum(x**2 for x in v))

def random_projection_gaussian(d, k, seed=0):
    """Gaussian random projection matrix (d -> k dimensions)."""
    rng = random.Random(seed)
    R = [[rng.gauss(0, 1/math.sqrt(k)) for _ in range(d)] for _ in range(k)]
    return R

def project(point, R):
    "Project point using projection matrix R (k x d)."
    return [sum(R[i][j]*point[j] for j in range(len(point))) for i in range(len(R))]

def dist(x, y):
    return math.sqrt(sum((a-b)**2 for a,b in zip(x,y)))

# Test distance preservation
rng = random.Random(7)
d = 500  # original dimension
n = 100  # number of points
X = [[rng.gauss(0,1) for _ in range(d)] for _ in range(n)]

epsilons = [0.3, 0.2, 0.1]
print(f"Original dim d={d}, n={n} points")


print(f"{'k':>6} {'ε_target':>10} {'Max distortion':>16} {'Frac within ε':>16}")

for eps in epsilons:
    k = jl_dimension(n, eps)
    R = random_projection_gaussian(d, k)
    Xp = [project(x, R) for x in X]

    distortions = []
    count_ok = 0; total = 0
    for i in range(min(n, 30)):
        for j in range(i+1, min(n, 30)):
            d_orig = dist(X[i], X[j])
            d_proj = dist(Xp[i], Xp[j])
            if d_orig > 0:
                ratio = d_proj / d_orig
                distortions.append(abs(ratio**2 - 1))
                total += 1
                if abs(ratio**2 - 1) <= eps: count_ok += 1
    max_dist = max(distortions) if distortions else 0
    frac_ok = count_ok / total if total > 0 else 0
    print(f"{k:>6} {eps:>10.1f} {max_dist:>16.4f} {frac_ok:>16.4f}")

Original dim d=500, n=100 points
     k   ε_target   Max distortion    Frac within ε


   512        0.3           0.1551           1.0000


  1063        0.2           0.1272           1.0000


  3948        0.1           0.0557           1.0000


## Sparse Random Projections (Achlioptas)

For very large $d$, even generating the $k \times d$ Gaussian matrix is expensive.

**Achlioptas (2001):** each entry of $R$ is:
$$R_{ij} = \sqrt{3} \times \begin{cases} +1 & \text{w.p. } 1/6 \\ 0 & \text{w.p. } 2/3 \\ -1 & \text{w.p. } 1/6 \end{cases}$$

- **Same JL guarantee** as Gaussian
- Only 1/3 of entries are non-zero → **3× faster** projection
- Can extend to 1/6 non-zeros with ±√6

**Very sparse projections** (Li et al. 2006): use $1/\sqrt{d}$ density → $O(\sqrt{d})$ work per point.

In [3]:
def sparse_random_matrix(d, k, sparsity=1/3, seed=0):
    """Sparse random projection matrix."""
    ""
    rng = random.Random(seed)
    R = []
    s = math.sqrt(3)
    p = sparsity / 2  # prob of +1 or -1
    for i in range(k):
        row = []
        for j in range(d):
            u = rng.random()
            if u < p: row.append(s)
            elif u < 2*p: row.append(-s)
            else: row.append(0.0)
        R.append(row)
    # Scale: E[R_ij^2] = 2p*3 = sparsity; need E=1/k overall
    scale = 1.0/math.sqrt(k * sparsity)
    return [[r*scale for r in row] for row in R]

k = jl_dimension(n, 0.2)
R_gauss  = random_projection_gaussian(d, k)
R_sparse = sparse_random_matrix(d, k)

Xg = [project(x, R_gauss) for x in X]
Xs = [project(x, R_sparse) for x in X]

# Compare distortion
pairs = [(i,j) for i in range(20) for j in range(i+1,20)]
g_errs = [abs((dist(Xg[i],Xg[j])/dist(X[i],X[j]))**2 - 1) for i,j in pairs if dist(X[i],X[j])>0]
s_errs = [abs((dist(Xs[i],Xs[j])/dist(X[i],X[j]))**2 - 1) for i,j in pairs if dist(X[i],X[j])>0]

print(f"k={k}, comparing Gaussian vs Sparse projection:")
print(f"  Gaussian: max_err={max(g_errs):.4f}, avg_err={sum(g_errs)/len(g_errs):.4f}")
print(f"  Sparse:   max_err={max(s_errs):.4f}, avg_err={sum(s_errs)/len(s_errs):.4f}")


k=1063, comparing Gaussian vs Sparse projection:
  Gaussian: max_err=0.1272, avg_err=0.0310
  Sparse:   max_err=2.2113, avg_err=1.9551
